# Multi agente simples

Neste exemplo, vamos criar um multi agente com um escritor e um revisor. O escritor irá gerar um texto com base em um prompt, e o revisor irá revisar o texto gerado pelo escritor, sugerindo melhorias.

## Carregando as dependências

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from random import randint
from typing import Annotated
from typing import cast

from agent_framework import tool
from pydantic import Field

## Carregando variáveis de ambiente

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
# print(f"AZURE_OPENAI_ENDPOINT: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
# print(f"AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: {os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")
# print(f"AZURE_OPENAI_API_VERSION: {os.getenv('AZURE_OPENAI_API_VERSION')}")

## Criando o cliente

Aqui mostramos algumas maneiras de inicializar o cliente do agente, usando diferentes tipos de chaves de API. O cliente é necessário para que o agente possa acessar as ferramentas e realizar suas tarefas.

In [ ]:
from agent_framework.azure import AzureOpenAIChatClient

# Usando variáveis de ambiente
# Set AZURE_OPENAI_ENDPOINT=""
# Set AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=""
# Set AZURE_OPENAI_API_VERSION=""
# Set AZURE_OPENAI_API_KEY=""
client = AzureOpenAIChatClient()

# # Ou passando os parâmetros diretamente
# client = AzureOpenAIChatClient(
#     endpoint="",
#     deployment_name="",
#     api_key=""
# )

# # Ou carregando a partir de um arquivo .env
# client = AzureOpenAIChatClient(
#     env_file_path="path/to/.env"
# )

## Criando os agentes

In [ ]:
writer_agent = client.as_agent(
    instructions=(
        "You are an excellent content writer. You create new content and edit contents based on the feedback."
    ),
    name="writer",
)

reviewer_agent = client.as_agent(
    instructions=(
        "You are an excellent content reviewer."
        "Provide actionable feedback to the writer about the provided content."
        "Provide the feedback in the most concise manner possible."
    ),
    name="reviewer",
)

## Criando o workflow

In [ ]:
from agent_framework import AgentResponse, WorkflowBuilder

# Build the workflow using the fluent builder.
# Set the start node via constructor and connect an edge from writer to reviewer.
workflow = WorkflowBuilder(start_executor=writer_agent).add_edge(writer_agent, reviewer_agent).build()

## Chamando o agente

In [ ]:
# Run the workflow with the user's initial message.
# For foundational clarity, use run (non streaming) and print the terminal event.
events = await workflow.run("Create a slogan for a new electric SUV that is affordable and fun to drive.")

In [ ]:
outputs = events.get_outputs()

# The outputs of the workflow are whatever the agents produce. So the outputs are expected to be a list
# of `AgentResponse` from the agents in the workflow.
outputs = cast(list[AgentResponse], outputs)
for output in outputs:
    print(f"{output.messages[0].author_name}: {output.text}\n")

In [ ]:
# Summarize the final run state (e.g., COMPLETED)
print("Final state:", events.get_final_state())